In [1]:
from neo4j import GraphDatabase,basic_auth
from neo4j_graphrag.schema import get_structured_schema, format_schema
import os
from nltk.corpus import stopwords
# import CyVer
# from CyVer import SyntaxValidator, PropertiesValidator, SchemaValidator
# from rapidfuzz import fuzz
# import regex as re
from sentence_transformers import SentenceTransformer,util
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import PorterStemmer
# import spacy
import torch
import re
from nltk.stem import WordNetLemmatizer
from transformers import AutoTokenizer, AutoModel
import sys
sys.path.append('../../')
from utils.KG_schema_formatting import format_final_schema

# # Get variables
url = os.getenv('SUSTAIN_DATABASE_URL')
user = os.getenv('SUSTAIN_DATABASE_USER')
password=os.getenv('SUSTAIN_DATABASE_PASSWORD')
database_name='neo4j'
driver = GraphDatabase.driver(url, auth=basic_auth(user, password), )
print("Connection established.",driver.verify_connectivity())

ModuleNotFoundError: No module named 'utils'

Here are the recommended naming conventions:

Labels: UpperCamelCase (a.k.a. PascalCase)

Relationships: CAPITALIZED_WITH_UNDERSCORE

Property key names: lowerCamelCase or snake_case


In [2]:
structured_schema = get_structured_schema(driver, is_enhanced=False, database = database_name, sanitize = False)
#  sanitize (bool): A flag to indicate whether to remove lists with
#             more than 128 elements from results. Useful for removing
#             embedding-like properties from database responses. Default is False.

In [ ]:
def preprocess_text(question:str) -> str:
        """Preprocess the text by removing special characters, converting to lowercase, and removing stopwords."""
        # Remove all the special characters and numbers
        question = re.sub(r'[^a-zA-Z\s]', ' ', str(question))
        # Converting to Lowercase
        question = question.lower()
        # Split into words
        tokens = question.split()
        #Remove stopwords
        stopWords = set(stopwords.words('english')) #a, and, the , be , your , ...
        tokens = [word for word in tokens if word not in stopWords]

        preprocessed_text = ' '.join(tokens)

        return preprocessed_text

def split_camel_case(s:str) -> list[str]:
    """Split camelCase or PascalCase into individual words. (FOR LABELS)"""
    return re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?![a-z])', s)


def split_by_underscore(s:str) -> list[str]:
    """Split snake_case into individual words.(for relationship types)"""
    return s.split('_')


def match_with_search_terms(search_terms: list[str], natural_language_question_term: str, match_substrings: bool) -> bool:
    """
    Checks if the single preprocessed word from the natural language question matches any of the search terms.
    
    - If `match_substrings` is True, it checks if any search term is a substring of the question term.
    - Otherwise, it checks for exact matches.
    """
    if match_substrings:
        #if the natural language question term is a substring of any search term ir vice versa return True
        return any(natural_language_question_term in term  for term in search_terms) or \
               any(term in natural_language_question_term for term in search_terms)
    else:
        #Check for exact match withot stemming
        exact_match = any(natural_language_question_term == term  for term in search_terms)
        if not exact_match:#Check for exact match with stemming
            stemmer = PorterStemmer()
            stemmed_question_term = stemmer.stem(natural_language_question_term)
            stemmed_search_terms = [stemmer.stem(term) for term in search_terms]
            exact_match =any(stemmed_question_term == term for term in stemmed_search_terms)
            if exact_match:
                print(f'Match found with stemming {natural_language_question_term} -> {stemmed_question_term} with {stemmed_search_terms}')
        return exact_match
        # return natural_language_question_term in search_terms


In [14]:
# #Load Spacy model
# spacy_nlp_model = spacy.load("en_core_web_lg")

# # Load bert model and tokenizer for word embeddings
# model_name = "google-bert/bert-base-uncased" # vector dimension is 768
# bert_tokenizer = AutoTokenizer.from_pretrained(model_name)
# bert_model = AutoModel.from_pretrained(model_name)
# bert_model.eval()
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
# bert_model = bert_model.to(device)

# # Function to get [CLS] token embedding of a word
# def get_cls_embedding(word):
#     #Tokenizes the input text and returns tensors formatted for PyTorch
#     inputs = bert_tokenizer(word, return_tensors="pt").to(device)
#     # Obtain the embeddings
#     with torch.no_grad(): #to avoid gradient calculations, which are not needed during inference.
#         outputs = bert_model(**inputs)
#     # Extract the last hidden state (embeddings)
#     cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token
#     return cls_embedding.cpu().numpy()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
model = model.to(device)
print(f"Using device: {device}")
def high_semantic_similarity( search_terms:list[str], natural_language_question_term:str) -> bool:
    # question_word_embedding = spacy_nlp_model(question_word)[0]
    # question_word_embedding = get_cls_embedding(question_word)
    # question_word_embedding = model.encode(natural_language_question_term)
    question_word_embedding = model.encode(natural_language_question_term, convert_to_tensor=True, device=device)#for gpu acceleration
    for search_term in search_terms:
        # search_term_embedding = get_cls_embedding(search_term)       
        # similarity_score = cosine_similarity(question_word_embedding, search_term_embedding)[0][0]
        # search_term_embedding = spacy_nlp_model(search_term)[0]
        # similarity_score = question_word_embedding.similarity(search_term_embedding)
        # search_term_embedding = model.encode(search_term)
        search_term_embedding = model.encode(search_term, convert_to_tensor=True, device=device)#for gpu acceleration
        similarity_score = round(util.cos_sim(question_word_embedding, search_term_embedding).item(),2) #for gpu
        if similarity_score >= 0.7:
            print(f"High semantic similarity found between '{natural_language_question_term}' and '{search_term}': {similarity_score:.2f}")
            return True
    return False

Using device: cuda


There are no vector representations for words that are not known and acronyms like DEGURBA, FUA, NUTS by using en_core_web_lg model !

In [15]:
def update_filtered_schema_with_nodes_rels(pruned_structure_dict,type_name, label, property_dict_list, one_hop_paths_list):
    ''' 
    It adds to the filtered KG schema the node label or rel type with its properties 
    and the one hop path, if any, that this node label or rel type is part of.
    If an one-hop path is added then all its node labels and rel types with their properties should be also added.
    Note: it checks only rels that have properties

    pruned_structure_dict: dict the dictionary from which we will construct the filtered schema
    type_name: str ('node_props','rel_props') defines if the label added will be node or rel
    label: str the node label or rel type to be added to the filtered schema
    property_dict_list: list of the properties of the input label (node label or rel type) 
    one_hop_paths_list: list of all the one hop paths of the KG schema
    
    It returns the updated pruned_structure_dict and the updated one_hop_paths_list.
    '''
    if label not in pruned_structure_dict[type_name]:
        pruned_structure_dict[type_name][label] = property_dict_list #add the label or rel type to the filtered schema with their properties
        #add the one hop paths they may be part of, and remove them form the list of one hop paths
        for relationship in one_hop_paths_list[:]:    
            if relationship['start'] == label or relationship['end'] == label or relationship['type'] == label:   
                #add the one-hop path to the filtered schema            
                pruned_structure_dict['relationships'].append({'start':relationship['start'], 'type':relationship['type'], 'end':relationship['end']})
                #remove this onehop path so i dont double check it in a next iteration
                one_hop_paths_list.remove(relationship)
                # add the nodes or rel that belog to the added one hop path
                # for node_label in [relationship['start'],  relationship['end']]:
                #     if (node_label not in pruned_structure_dict['node_props']) and (node_label in structured_schema['node_props']):
                #         pruned_structure_dict['node_props'][node_label] = structured_schema['node_props'][node_label] #add the label (if it haas properties) to the filtered schema with their properties
                # rel_type = relationship['type']
                # if (rel_type not in pruned_structure_dict['rel_props']) and (rel_type in structured_schema['rel_props']):
                #     pruned_structure_dict['rel_props'][rel_type] = structured_schema['rel_props'][rel_type] #add the rel type (If it has properties) to the filtered schema with their properties


    return pruned_structure_dict, one_hop_paths_list

def update_filtered_schema_with_one_hop_paths(pruned_structure_dict,type_name, relationship):
    ''' 
    It adds to the filtered KG schema the one hop paths that their rel type was found in the schema.
    It also adds the nodes with their properties of these one hop paths to the filtered KG schema, if they are not already.

    pruned_structure_dict: dict the dictionary from which we will construct the filtered schema
    type_name: str ('relationships') Defines that one-hop-paths will be added to the filtered schema
    relationship: dict the relationship (one-hop-path) to be added to the filtered schema

    It returns the updated pruned_structure_dict and the updated one_hop_paths_list.
    '''
    
    pruned_structure_dict[type_name].append({'start':relationship['start'], 'type':relationship['type'], 'end':relationship['end']})
    # #Check the node labels of the one hop path to be added
    # for node_label in [relationship['start'],  relationship['end']]:
    #     if (node_label not in pruned_structure_dict['node_props']) and (node_label in structured_schema['node_props']):
    #         pruned_structure_dict['node_props'][node_label] = structured_schema['node_props'][node_label] #add the label (if it haas properties) to the filtered schema with their properties
    return pruned_structure_dict


# def get_filtered_schema_by_question(natural_language_question, structured_schema, match_substrings:bool = False, check_semantic_similarity:bool = False) -> dict:
#     '''
#     It returns a filtered schema of the KG based on the natural language question.
#     The filtering is done by checking if the node labels or rel types are present in the question.
#     If a node label or rel type is found in the question, then they are added with their properties to the filtered schema.
#     Also the one hop paths the may be part of are added to the schema.
#     If a rel type of a one hop path is found in the question (that was not found by checking rel types if they had no poroperties), the one hop path is added to the filtered schema.

    
#     It searches for each node label and rel type and one hop path rel if they are mentioned in the question.
    
#     natural_language_question: str the natural language question to be used for filtering
#     structured_schema: dict the structured schema of the KG
#     match_substrings: bool if True, it matches substrings in the question (default is False)
#     semantic_similarity: bool if True, it uses semantic similarity to match words in the question (default is False)

#     It returns the filtered schema of the KG.
#     '''
    
#     # Initialize an empty dictionary to store the filtered schema
#     pruned_structure_dict = {
#         'node_props': {},
#         'rel_props': {},
#         'relationships': []
#     }
    
#     one_hop_paths_list = structured_schema['relationships'].copy()
#     #  the question is preprocessed to remove special characters, convert to lowercase and remove stopwords and split into words
#     natural_language_question_terms = preprocess_text(natural_language_question).split()
#     # Iterate over all nodes and relationships with properties in the structured schema
#     for type_name in ['node_props','rel_props']:
#         for label , property_dict_list in structured_schema[type_name].items():
#             split_labels = split_camel_case(label) #split the label into words if it is camelCase or PascalCase and  add them as 2 seperate search terms
#             search_terms = split_labels #+ [label] #add also the combined label????????
#             #------->Should i use the properties as search terms? (if the quesrtion does not containany node labels or relationship types)
#             # search_terms = [label] #+ [property_dict["property"] for property_dict in property_dict_list]
#             # Check if any of the search terms are present in the natural language question
#             if contains_search_terms(search_terms, natural_language_question_terms,match_substrings=match_substrings): #it matches whole words not substrings
#                 print(f"Found exact label: {label} in question with search term {search_terms}")
#                 pruned_structure_dict, one_hop_paths_list = update_filtered_schema_with_nodes_rels(pruned_structure_dict,type_name, label, property_dict_list, one_hop_paths_list)
                
#             elif check_semantic_similarity and high_semantic_similarity(search_terms, natural_language_question_terms):         
#                 #Check for semantic similarity with embeddings
#                 print(f"Found by semantic similarity: {label} in question with search term {search_terms}")
#                 pruned_structure_dict, one_hop_paths_list = update_filtered_schema_with_nodes_rels(pruned_structure_dict,type_name, label, property_dict_list, one_hop_paths_list)
                
#     type_name = 'relationships'
#     for relationship in one_hop_paths_list:
#         # check if the relationship is split by underscore, so it will be added as 2 search terms
#         # i dont have to search again for labels, only the relationship types that i may have not checked if they didnt have properties
#         search_terms = split_by_underscore(relationship['type'])
        
#         if contains_search_terms(search_terms, natural_language_question_terms, match_substrings=match_substrings):
#             print(f"Found relationship: {relationship['type']} from one hop path {relationship} in question with search term {search_terms}")
#             pruned_structure_dict = update_filtered_schema_with_one_hop_paths(pruned_structure_dict,type_name, relationship)
        
#         elif check_semantic_similarity and high_semantic_similarity(search_terms, natural_language_question_terms): 
#             #Check for semantic similarity with embeddings
#             print(f"Found by semantic similarity: {label} in question with search term {search_terms}")
#             pruned_structure_dict = update_filtered_schema_with_one_hop_paths(pruned_structure_dict,type_name, relationship)
    
#     #  Format the structured schema into a human-readable string.
#     filtered_schema = format_schema(pruned_structure_dict, is_enhanced=False)
#     return filtered_schema

In [18]:
def get_filtered_schema_by_question(driver, database_name, natural_language_question, match_substrings:bool = False, check_semantic_similarity:bool = False, inline_props:bool = True) -> str:
    '''
    It returns a filtered schema of the KG based on the natural language question.
    The filtering is done by checking if the node labels or rel types are present in the question.
    If a node label or rel type is found in the question, then they are added with their properties to the filtered schema.
    Also the one hop paths the may be part of are added to the schema.
    If a rel type of a one hop path is found in the question (that was not found by checking rel types if they had no properties), the one hop path is added to the filtered schema.
    
    It searches for each term in the natural language question if they match exactly or semantically with one or more node labels, rel types or one hop paths rels.
    
    driver: the neo4j driver of the KG database
    database_name(str): the database name
    natural_language_question: str the natural language question to be used for filtering
    structured_schema: dict the structured schema of the KG
    match_substrings: bool if True, it matches substrings in the question (default is False)
    semantic_similarity: bool if True, it uses semantic similarity to match words in the question (default is False)
    inline_props : bool If True, includes property types inline within one-hop path descriptions in the returned filtered schema. 
                             If False, lists the node/rel definitions separately.
    
    It returns the filtered schema of the KG.
    '''
    
    # Initialize an empty dictionary to store the filtered schema
    pruned_structure_dict = {
        'node_props': {},
        'rel_props': {},
        'relationships': []
    }
    #get the structured schema of the KG
    structured_schema = get_structured_schema(driver, is_enhanced = False, database = database_name, sanitize = False)

    one_hop_paths_list = structured_schema['relationships'].copy()
    #  the question is preprocessed to remove special characters, convert to lowercase and remove stopwords and split into words
    natural_language_question_terms = list(set(preprocess_text(natural_language_question).split()))

    for term in natural_language_question_terms:
        # Check if the term is a node label or rel type in the structured schema

        print(f"Checking term: {term}")
        for type_name in ['node_props','rel_props']:
            for label , property_dict_list in structured_schema[type_name].items():
                split_labels = split_camel_case(label)
                search_terms = split_labels + [label] if len(split_labels) > 1 else [label] #add also the original label in search terms if it was split
                #------->Should i use the properties as search terms? (if the quesrtion does not containany node labels or relationship types)
                # search_terms = [label] #+ [property_dict["property"] for property_dict in property_dict_list]
                search_terms = [term.lower() for term in search_terms]
                if match_with_search_terms(search_terms, term, match_substrings=match_substrings):
                    print(f"Found exact label: {label} in question with search term {search_terms}")
                    pruned_structure_dict, one_hop_paths_list = update_filtered_schema_with_nodes_rels(pruned_structure_dict,type_name, label, property_dict_list, one_hop_paths_list)
                    
                elif check_semantic_similarity and high_semantic_similarity(search_terms, term):
                    #Check for semantic similarity with embeddings
                    print(f"Found by semantic similarity: {label} in question with search term {search_terms}")
                    pruned_structure_dict, one_hop_paths_list = update_filtered_schema_with_nodes_rels(pruned_structure_dict,type_name, label, property_dict_list, one_hop_paths_list)
            
        ## Check if the term is a relationship type in the structured schema
        type_name = 'relationships'
        for relationship in one_hop_paths_list:
            # check if the relationship is split by underscore, so it will be added as 2 search terms
            # i dont have to search again for labels, only the relationship types that i may have not checked if they didnt have properties
            split_terms = split_by_underscore(relationship['type'])
            search_terms = split_terms + [relationship['type']] if len(split_terms) > 1 else [relationship['type']] #add lasi the original label in search terms if it was split
            search_terms = [term.lower() for term in search_terms]           
            if match_with_search_terms(search_terms, term, match_substrings=match_substrings):
                print(f"Found relationship: {relationship['type']} from one hop path {relationship} in question with search term {search_terms}")
                pruned_structure_dict = update_filtered_schema_with_one_hop_paths(pruned_structure_dict,type_name, relationship)
                
            elif check_semantic_similarity and high_semantic_similarity(search_terms,term): 
                #Check for semantic similarity with embeddings
                print(f"Found by semantic similarity: {label} in question with search term {search_terms}")
                pruned_structure_dict = update_filtered_schema_with_one_hop_paths(pruned_structure_dict,type_name, relationship)
    
    #  Format the structured schema into a human-readable string.
    filtered_schema = format_final_schema(driver, database_name, pruned_structure_dict, structured_schema, inline_props=inline_props)
    # filtered_schema = format_schema(pruned_structure_dict, is_enhanced=False)
    return filtered_schema


In [17]:
natural_language_question = "area?"
print('Question:\n', natural_language_question)
print('Search terms in words:\n', list(set(preprocess_text(natural_language_question).split())))
#Get filtered schema based on question
filtered_schema = get_filtered_schema_by_question(driver, database_name, natural_language_question, match_substrings = False, check_semantic_similarity = True, inline_props= True)
print(filtered_schema)

Question:
 area?
Search terms in words:
 ['area']
Checking term: area
Found exact label: GeoArea in question with search term ['geo', 'area', 'geoarea']
Found exact label: Area in question with search term ['area']
High semantic similarity found between '['area']' and 'region': 0.72
Found by semantic similarity: SubRegion in question with search term ['sub', 'region', 'subregion']
High semantic similarity found between '['area']' and 'region': 0.72
Found by semantic similarity: Region in question with search term ['region']
Found exact label: REFERS_TO_AREA in question with search term ['refers', 'to', 'area', 'refers_to_area']
Node properties:
GeoArea {name: STRING, M49code: STRING, ISOalpha2code: STRING, ISOalpha3code: STRING, EUcode: STRING}
Area {name: STRING, M49code: STRING, ISOalpha2code: STRING, ISOalpha3code: STRING, EUcode: STRING}
SubRegion {M49code: STRING, name: STRING}
Region {M49code: STRING, name: STRING}
Relationship properties:
REFERS_TO_AREA {point: POINT, csrNumber:

In [19]:
natural_language_question = "area?"
print('Question:\n', natural_language_question)
print('Search terms in words:\n', list(set(preprocess_text(natural_language_question).split())))
#Get filtered schema based on question
filtered_schema = get_filtered_schema_by_question(driver, database_name, natural_language_question, match_substrings = False, check_semantic_similarity = True, inline_props= True)
print(filtered_schema)

Question:
 area?
Search terms in words:
 ['area']
Checking term: area
Found exact label: GeoArea in question with search term ['geo', 'area', 'geoarea']
Found exact label: Area in question with search term ['area']
High semantic similarity found between 'area' and 'region': 0.72
Found by semantic similarity: SubRegion in question with search term ['sub', 'region', 'subregion']
High semantic similarity found between 'area' and 'region': 0.72
Found by semantic similarity: Region in question with search term ['region']
Found exact label: REFERS_TO_AREA in question with search term ['refers', 'to', 'area', 'refers_to_area']
Node properties:
GeoArea {name: STRING, M49code: STRING, ISOalpha2code: STRING, ISOalpha3code: STRING, EUcode: STRING}
Area {name: STRING, M49code: STRING, ISOalpha2code: STRING, ISOalpha3code: STRING, EUcode: STRING}
SubRegion {M49code: STRING, name: STRING}
Region {M49code: STRING, name: STRING}
Relationship properties:
REFERS_TO_AREA {point: POINT, csrNumber: INTEGER

## New Schema Filtering

In [1]:
import sys
from nltk.stem import PorterStemmer
from neo4j import GraphDatabase,basic_auth
import pandas as pd
from sentence_transformers import SentenceTransformer,util
from neo4j_graphrag.schema import get_structured_schema, get_schema
import torch
# This adds the parent directory (../) to the list of directories Python will search for modules.
# The ../ refers to the directory one level up from the current working directory.
sys.path.append('../')
from utils.KG_schema_formatting import format_final_schema,get_structured_schema_custom
from utils.nlp_functions import preprocess_text,split_camel_case,split_by_underscore

#Define model for word semantic similarity
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

In [2]:
   
def match_with_search_terms(search_terms: list[str], natural_language_question_term: str, match_substrings: bool = False) -> bool:
    """
    Checks if the single preprocessed word from the natural language question matches any of the search terms.
    
    - If `match_substrings` is True, it checks if any search term is a substring of the question term.
    - Otherwise, it checks for exact matches.
    """
    if match_substrings:
        #if the natural language question term is a substring of any search term ir vice versa return True
        return any(natural_language_question_term in term  for term in search_terms) or \
               any(term in natural_language_question_term for term in search_terms)
    else:
        #Check for exact match withot stemming
        exact_match = any(natural_language_question_term == term  for term in search_terms)
        if not exact_match:#Check for exact match with stemming
            stemmer = PorterStemmer()
            stemmed_question_term = stemmer.stem(natural_language_question_term)
            stemmed_search_terms = [stemmer.stem(term) for term in search_terms]
            exact_match =any(stemmed_question_term == term for term in stemmed_search_terms)
            if exact_match:
                print(f'Match found with stemming {natural_language_question_term} -> {stemmed_question_term} with {stemmed_search_terms}')
        return exact_match
        # return natural_language_question_term in search_terms

def batch_encode_terms(model, terms: list[str], device):
    '''
    Encode a list of terms into embeddings using the sentence transformer model
    Returns a dict with terms as keys and their embeddings as values
    '''
    embeddings = model.encode(terms, convert_to_tensor=True, device=device)
    return dict(zip(terms, embeddings))

def high_semantic_similarity(search_terms: list[str], natural_language_question_term: str, embeddings,embeddings_keys):
    ''' 
    Check if the question term is semantically similar to any of the search terms using cosine similarity
    Returns True if any term has a cosine similarity >= 0.7 with the question term
    '''
    filtered_terms = [t for t in search_terms if t in embeddings_keys]

    if natural_language_question_term not in embeddings_keys or not filtered_terms:
        return False

    return any(
        round(util.cos_sim(embeddings[natural_language_question_term], embeddings[t]).item(),2) >= 0.7
        for t in filtered_terms
    )

def update_filtered_schema_with_nodes_rels(pruned_structure_dict,type_name, label, property_dict_list, one_hop_paths_list):
    ''' 
    It adds to the filtered KG schema the node label or rel type with its properties 
    and the one hop path, if any, that this node label or rel type is part of.
    Note: it checks only rels that have properties

    pruned_structure_dict: dict the dictionary from which we will construct the filtered schema
    type_name: str ('node_props','rel_props') defines if the label added will be node or rel
    label: str the node label or rel type to be added to the filtered schema
    property_dict_list: list of the properties of the input label (node label or rel type) 
    one_hop_paths_list: list of all the one hop paths of the KG schema
    
    It returns the updated pruned_structure_dict and the updated one_hop_paths_list.
    '''
    if label not in pruned_structure_dict[type_name]:
        pruned_structure_dict[type_name][label] = property_dict_list #add the label or rel type to the filtered schema with their properties
        #add the one hop paths they may be part of, and remove them form the list of one hop paths
        for relationship in one_hop_paths_list[:]:    
            if relationship['start'] == label or relationship['end'] == label or relationship['type'] == label:   
                #add the one-hop path to the filtered schema            
                pruned_structure_dict['relationships'].append({'start':relationship['start'], 'type':relationship['type'], 'end':relationship['end']})
                #remove this onehop path so i dont double check it in a next iteration
                one_hop_paths_list.remove(relationship)
                # # add the nodes or rel that belog to the added one hop path
                # for node_label in [relationship['start'],  relationship['end']]:
                #     if (node_label not in pruned_structure_dict['node_props']) and (node_label in structured_schema['node_props']):
                #         pruned_structure_dict['node_props'][node_label] = structured_schema['node_props'][node_label] #add the label (if it haas properties) to the filtered schema with their properties
                # rel_type = relationship['type']
                # if (rel_type not in pruned_structure_dict['rel_props']) and (rel_type in structured_schema['rel_props']):
                #     pruned_structure_dict['rel_props'][rel_type] = structured_schema['rel_props'][rel_type] #add the rel type (If it has properties) to the filtered schema with their properties


    return pruned_structure_dict, one_hop_paths_list

def update_filtered_schema_with_one_hop_paths(pruned_structure_dict,type_name, relationship):
    ''' 
    It adds to the filtered KG schema the one hop paths that their rel type was found in the schema.

    pruned_structure_dict: dict the dictionary from which we will construct the filtered schema
    type_name: str ('relationships') Defines that one-hop-paths will be added to the filtered schema
    relationship: dict the relationship (one-hop-path) to be added to the filtered schema
   
    It returns the updated pruned_structure_dict and the updated one_hop_paths_list.
    '''
    
    pruned_structure_dict[type_name].append({'start':relationship['start'], 'type':relationship['type'], 'end':relationship['end']})
    # #Check the node labels of the one hop path to be added
    # for node_label in [relationship['start'],  relationship['end']]:
    #     if (node_label not in pruned_structure_dict['node_props']) and (node_label in structured_schema['node_props']):
    #         pruned_structure_dict['node_props'][node_label] = structured_schema['node_props'][node_label] #add the label (if it haas properties) to the filtered schema with their properties
    return pruned_structure_dict

def get_filtered_schema_by_question(driver, database_name, natural_language_question,
                                          match_substrings=False,
                                          check_semantic_similarity=False,
                                          inline_props=True):
    '''
    It returns a filtered schema of the KG based on the natural language question.
    The filtering is done by checking if the node labels or rel types are present in the question.
    If a node label or rel type is found in the question, then they are added with their properties to the filtered schema.
    Also the one hop paths the may be part of are added to the schema.
    If a rel type of a one hop path is found in the question (that was not found by checking rel types if they had no properties), the one hop path is added to the filtered schema.
    
    It searches for each term in the natural language question if they match exactly or semantically with one or more node labels, rel types or one hop paths rels.
    
    driver: the neo4j driver of the KG database
    database_name(str): the database name
    natural_language_question: str the natural language question to be used for filtering
    structured_schema: dict the structured schema of the KG
    match_substrings: bool if True, it matches substrings in the question (default is False)
    semantic_similarity: bool if True, it uses semantic similarity to match words in the question (default is False)
    inline_props : bool If True, includes property types inline within one-hop path descriptions in the returned filtered schema. 
                             If False, lists the node/rel definitions separately.
    
    It returns the filtered schema of the KG.
    '''
    # Initialize an empty dictionary to store the filtered schema
    pruned_structure_dict = {
        'node_props': {},
        'rel_props': {},
        'relationships': []
    }

    # Get the full structured schema from the database
    structured_schema = get_structured_schema(driver, is_enhanced=False, database=database_name, sanitize=False)
    one_hop_paths_list = structured_schema['relationships'].copy()

     # The question is preprocessed to remove special characters, 
     # convert to lowercase and remove stopwords and split into words
    natural_language_question_terms = list(set(preprocess_text(natural_language_question).split()))
    natural_language_question_terms_lower = [t.lower() for t in natural_language_question_terms]

    # Collect all terms from the schema to embed
    all_kg_terms = set()

    # Nodes and Relationships
    for type_name in ['node_props', 'rel_props']:
        for label in structured_schema[type_name].keys():
            splits = split_camel_case(label)
             #add the original label in search terms if it was split
            if len(splits) > 1:
                all_kg_terms.update(s.lower() for s in (splits + [label]))
            else:
                all_kg_terms.add(label.lower())

    # One Hop Paths
    for rel in one_hop_paths_list:
        splits = split_by_underscore(rel['type'])
         #add the original label in search terms if it was split
        if len(splits) > 1:
            all_kg_terms.update(s.lower() for s in (splits + [rel['type']]))
        else:
            all_kg_terms.add(rel['type'].lower())

    # Include question terms to ensure they’re also embedded
    all_terms_to_encode = list(all_kg_terms.union(natural_language_question_terms_lower))

    # Batch encode all required terms once
    embeddings = batch_encode_terms(model, all_terms_to_encode, device)

    embeddings_keys = set(embeddings.keys())  # For quick lookup
    # Cache semantic match results to avoid recomputation
    semantic_match_cache = {}  

    # Process each term in the question
    for term, term_lower in zip(natural_language_question_terms, natural_language_question_terms_lower):
        print(f"Checking term: {term } (lower: {term_lower})")

        # Check against node and relationship properties
        for type_name in ['node_props', 'rel_props']:
            for label, property_dict_list in structured_schema[type_name].items():
                split_labels = split_camel_case(label)
                search_terms = split_labels + [label] if len(split_labels) > 1 else [label]
                search_terms_lower = [s.lower() for s in search_terms]
                # Try exact or substring match
                if match_with_search_terms(search_terms_lower, term_lower, match_substrings=match_substrings):
                    print(f"Found exact label: {label} in question with search term {search_terms_lower}")
                    pruned_structure_dict, one_hop_paths_list = update_filtered_schema_with_nodes_rels(
                        pruned_structure_dict, type_name, label, property_dict_list, one_hop_paths_list)

                # Elif not exact match, check for high semantic similarity match(if True)
                elif check_semantic_similarity:
                    key = (tuple(search_terms_lower), term_lower)
                    if key not in semantic_match_cache:
                        semantic_match_cache[key] = high_semantic_similarity(search_terms_lower, term_lower, embeddings,embeddings_keys)
                    if semantic_match_cache[key]:
                        print(f"Found by semantic similarity: {label} in question with search term {search_terms_lower}")
                        pruned_structure_dict, one_hop_paths_list = update_filtered_schema_with_nodes_rels(
                            pruned_structure_dict, type_name, label, property_dict_list, one_hop_paths_list)

        # Now check relationships (one-hop paths)
        type_name = 'relationships'
        for relationship in one_hop_paths_list[:]:  # Copy for safe removal
            split_terms = split_by_underscore(relationship['type'])
            search_terms = split_terms + [relationship['type']] if len(split_terms) > 1 else [relationship['type']]
            search_terms_lower = [s.lower() for s in search_terms]


            # Try exact or substring match
            if match_with_search_terms(search_terms_lower, term_lower, match_substrings=match_substrings):
                print(f"Found relationship: {relationship['type']} from one hop path {relationship} in question with search term {search_terms_lower}")
                pruned_structure_dict = update_filtered_schema_with_one_hop_paths(pruned_structure_dict, type_name, relationship)

            # Elif not exact match, check for high semantic similarity match(if True)
            elif check_semantic_similarity:
                key = (tuple(search_terms_lower), term_lower)
                if key not in semantic_match_cache:
                    semantic_match_cache[key] = high_semantic_similarity(search_terms_lower, term_lower, embeddings,embeddings_keys)
                if semantic_match_cache[key]:
                    print(f"Found by semantic similarity: {relationship['type']} in question with search term {search_terms_lower}")
                    pruned_structure_dict = update_filtered_schema_with_one_hop_paths(pruned_structure_dict, type_name, relationship)
    print(structured_schema['rel_props']['similar_company_as'])
    # Format and return the final filtered schema
    filtered_schema = format_final_schema(driver, database_name, pruned_structure_dict, structured_schema, inline_props=inline_props)
    return filtered_schema


In [2]:

test_data = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
#update df only with rows with a database access
test_data = test_data[test_data['url'].notna()].reset_index(drop=True)

# for i,row in test_data.iterrows():
i = 10
row = test_data.iloc[i]  # Access the row directly
question = row['question']
print(i, question)
database_url = row['url']
database_name = row['database_name']
database_user = row['username']
database_password = row['password']
print('--------------------------------------------')
#Connect to the database
driver = GraphDatabase.driver(database_url, auth=basic_auth(database_user, database_password) )

# print(get_schema(driver, is_enhanced=False, database=database_name, sanitize=False))
#Get filtered schema based on question
# filtered_schema = get_filtered_schema_by_question(driver, database_name, question, match_substrings = False, check_semantic_similarity = True, inline_props= True)
# print('--------------------------------------------')
# print(filtered_schema)
# print('--------------------------------------------')
# print(row['cypher'])


10 List entities that have undergone a company type change from the original registration.
--------------------------------------------


In [6]:
for i in get_structured_schema(driver)['rel_props']:
    if i not in get_structured_schema_custom(driver, is_enhanced=False, database=database_name, sanitize=False)['rel_props']:
        print(i)

CypherSyntaxError: {code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input '=': expected ':' (line 1, column 35 (offset: 34))
"CALL apoc.meta.data(config={sample=-1}) YIELD label, other, elementType, type, property WHERE NOT type = 'RELATIONSHIP' AND elementType = 'node' AND NOT label IN $EXCLUDED_LABELS WITH label AS nodeLabel, collect({property:property, type:type}) AS properties RETURN {label: nodeLabel, properties: properties} AS output"
                                   ^}

In [4]:
get_structured_schema_custom(driver)['rel_props']

{'registered_address': [{'property': 'link', 'type': 'STRING'},
  {'property': 'sourceID', 'type': 'STRING'},
  {'property': 'valid_until', 'type': 'STRING'}],
 'intermediary_of': [{'property': 'valid_until', 'type': 'STRING'},
  {'property': 'sourceID', 'type': 'STRING'},
  {'property': 'link', 'type': 'STRING'},
  {'property': 'end_date', 'type': 'STRING'},
  {'property': 'start_date', 'type': 'STRING'},
  {'property': 'lastEditTimestamp', 'type': 'STRING'}],
 'officer_of': [{'property': 'sourceID', 'type': 'STRING'},
  {'property': 'link', 'type': 'STRING'},
  {'property': 'valid_until', 'type': 'STRING'},
  {'property': 'start_date', 'type': 'STRING'},
  {'property': 'end_date', 'type': 'STRING'},
  {'property': 'status', 'type': 'STRING'},
  {'property': 'lastEditTimestamp', 'type': 'STRING'}],
 'underlying': [{'property': 'link', 'type': 'STRING'},
  {'property': 'valid_until', 'type': 'STRING'},
  {'property': 'sourceID', 'type': 'STRING'}],
 'similar': [{'property': 'link', 'ty

In [16]:

get_structured_schema_custom(driver, is_enhanced=False, database=database_name, sanitize=False)['rel_props'].keys()

dict_keys(['registered_address', 'intermediary_of', 'officer_of', 'underlying', 'similar', 'same_as', 'same_name_as', 'same_company_as', 'same_id_as', 'connected_to'])

## Create embeddings for node labels and rel types per database

In [ ]:
import pandas as pd
import json
import sys 
sys.path.append('../../')
from utils.nlp_functions import preprocess_text,split_by_underscore,split_camel_case

In [ ]:

df = pd.read_excel('../../data/text2cypher-2025v1/test.xlsx')
db_credentials_df = df[['url','database_name','username','password']].dropna().drop_duplicates()
# Create list of dictionaries
db_credentials = [
    {
        "name": row['database_name'],
        "url": row['url'],
        "user": row['username'],
        "password": row['password']
    }
    for _, row in db_credentials_df.iterrows()
]

db_credentials.append( {
        "name":os.getenv('database_name'),
        "url":os.getenv('database_url'),
        "user": os.getenv('database_user'),
        "password": os.getenv('database_password')
    })

def save_embeddings(db_name, KG_term_embeddings, schema_file='kg_term_embeddings.json'):
    '''
    Save or update the KG term embeddings for a specific db_name into a JSON file.

    Args:
        db_name (str): The name of the database whose dict with terms and embeddings is to be saved.
        KG_term_embeddings (dict): The mapping of terms and embeddings to save.
        schema_file (str): Path to the JSON file to update.
    '''
    # Load existing schemas if the file exists
    if os.path.exists(schema_file):
        with open(schema_file, 'r') as f:
            try:
                kg_term_embeddings = json.load(f)
            except json.JSONDecodeError:
                kg_term_embeddings = {}
    else:
        kg_term_embeddings = {}

    # Update the schema for the specific db_name
    kg_term_embeddings[db_name] = KG_term_embeddings

    # Save updated schemas back to file
    with open(schema_file, 'w') as f:
        json.dump(kg_term_embeddings, f, indent=2)

In [26]:
# Setup device and model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
model = model.to(device)
print(f"Using device: {device}")


for db in db_credentials:
    # Connect to the database
    url = db['url']
    user = db['user']
    password = db['password']
    database_name = db['name']
    driver = GraphDatabase.driver(url, auth=basic_auth(user, password), )
    print(f"Connection established to {database_name} database.",driver.verify_connectivity())
    
        
    # Step 1: Prepare a set of unique terms in the KG schema
    all_search_terms = set()

    # From structured schema
    structured_schema = get_structured_schema(driver, is_enhanced=False, database=database_name, sanitize=False)
    for type_name in ['node_props', 'rel_props']:
        for label, property_dict_list in structured_schema[type_name].items():
            split_labels = split_camel_case(label)
            search_terms = split_labels + [label] if len(split_labels) > 1 else [label]
            search_terms = [term.lower() for term in search_terms]
            all_search_terms.update(search_terms)

    # From relationships
    for relationship in structured_schema['relationships']:
        split_terms = split_by_underscore(relationship['type'])
        search_terms = split_terms + [relationship['type']] if len(split_terms) > 1 else [relationship['type']]
        search_terms = [term.lower() for term in search_terms]
        all_search_terms.update(search_terms)

    # Step 2: Encode all search terms once and store in a dictionary
    KG_term_embeddings = {}
    for term in all_search_terms:
        print(f"Encoding term: {term}")
        embedding = model.encode(term, convert_to_tensor=True, device=device)
        #Convert tensor embedding to list to save to json
        embedding_list = embedding.cpu().tolist()
        KG_term_embeddings[term] = embedding_list
    #Saving the dict of embedding for each label and rel type in a dictionary per database name
    save_embeddings(database_name, KG_term_embeddings, schema_file='kg_term_embeddings.json')


Using device: cuda
Connection established to northwind database. None
Encoding term: orders
Encoding term: supplier
Encoding term: part
Encoding term: order
Encoding term: product
Encoding term: of
Encoding term: part_of
Encoding term: supplies
Encoding term: customer
Encoding term: category
Encoding term: purchased
Connection established to neoflix database. None
Encoding term: original_language
Encoding term: video
Encoding term: genre
Encoding term: subscription
Encoding term: productioncompany
Encoding term: in
Encoding term: cast_for
Encoding term: user
Encoding term: spoken
Encoding term: movie
Encoding term: produced_by
Encoding term: package
Encoding term: in_genre
Encoding term: person
Encoding term: spoken_in_language
Encoding term: by
Encoding term: country
Encoding term: has_keyword
Encoding term: to
Encoding term: adult
Encoding term: original
Encoding term: produced
Encoding term: production
Encoding term: crew_for
Encoding term: provides
Encoding term: language
Encoding 

In [ ]:
#convert from list to tensor
embedding_tensor = torch.tensor(embedding_list)

## Embeddings for questions

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

In [ ]:
datasets={
    'neo4j': '../../data/text2cypher-2025v1/test.xlsx',
    'sustain': '../../data/text2cypher-sustaingraph/test.xlsx',
}
for dataset in datasets:
    if dataset =='sustain':
        df = pd.read_excel(datasets['sustain'],sheet_name='Queries')     
    elif dataset == 'neo4j':
        df = pd.read_excel(datasets['neo4j'])
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
    for i,row in df.iterrows():
        # Get the question
        if  dataset =='sustain':
            natural_language_question = row['Question']
        else:
            natural_language_question = row['question']
        # Preprocess the question to remove special characters, convert to lowercase and remove stopwords and split into words
        natural_language_question_terms = list(set(preprocess_text(natural_language_question).split()))
        for term in natural_language_question_terms:
            # calculate the embedding for the term if not already calculated and save it to the json file
            


# Compare the models for word embeddings

In [ ]:
# Load bert model and tokenizer for word embeddings
model_name = "google-bert/bert-base-uncased" # vector dimension is 768
bert_tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)
bert_model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
bert_model = bert_model.to(device)

# Function to get [CLS] token embedding of a word
def get_cls_embedding(word):
    #Tokenizes the input text and returns tensors formatted for PyTorch
    inputs = bert_tokenizer(word, return_tensors="pt").to(device)
    # Obtain the embeddings
    with torch.no_grad(): #to avoid gradient calculations, which are not needed during inference.
        outputs = bert_model(**inputs)
    # Extract the last hidden state (embeddings)
    cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token
    return cls_embedding.cpu().numpy()


def compare_similarity_models(word1, word2):
    
    def calculate_similarity(embedding1, embedding2):
        similarity = cosine_similarity([embedding1], [embedding2])[0][0]
        return similarity

    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    embeddings = model.encode([word1, word2])
    similarity =  calculate_similarity(embeddings[0], embeddings[1])
    print(f"Similarity with sentence-transformers between '{word1}' and '{word2}': {similarity:.4f}")

    #----------------------------------
    
    spacy_nlp_model = spacy.load("en_core_web_lg")
    # Process the words
    token1 = spacy_nlp_model(word1)[0]
    token2 = spacy_nlp_model(word2)[0]
    # Compute similarity
    similarity = token1.similarity(token2)
    print(f"Similarity with SpaCy between '{word1}' and '{word2}': {similarity:.4f}")


    #----------------------------------
    #https://medium.com/@davidlfliang/intro-getting-started-with-text-embeddings-using-bert-9f8c3b98dee6 
    
    # Get embeddings
    embedding_have = get_cls_embedding(word1)
    embedding_own = get_cls_embedding(word2)

    # Compute cosine similarity
    similarity = cosine_similarity(embedding_have, embedding_own)
    print(f"Similarity with BERT between '{word1}' and '{word2}':{similarity[0][0]:.4f}")


In [161]:
compare_similarity_models("hazard", "hazards")

Similarity with sentence-transformers between 'hazard' and 'hazards': 0.6008
Similarity with SpaCy between 'hazard' and 'hazards': 0.8178
Similarity with BERT between 'hazard' and 'hazards':0.9718


Comments: .
BERT return a high similarity (0.9) for every single word combo(even completely irrelevant)
SpaCy failed to return high similarity between european-europeans (0.54)


Similarity with sentence-transformers between 'country' and 'geoarea': 0.4408
Similarity with SpaCy between 'country' and 'geoarea': 0.0000
Similarity with BERT between 'country' and 'geoarea':0.8412

Similarity with sentence-transformers between 'country' and 'area': 0.4285
Similarity with SpaCy between 'country' and 'area': 0.4437
Similarity with BERT between 'country' and 'area':0.8847

Similarity with sentence-transformers between 'country' and 'nation': 0.7664
Similarity with SpaCy between 'country' and 'nation': 0.7488
Similarity with BERT between 'country' and 'nation':0.9280

Similarity with sentence-transformers between 'country' and 'region': 0.5304
Similarity with SpaCy between 'country' and 'region': 0.5427
Similarity with BERT between 'country' and 'region':0.8871